In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from ellipse import *
from utils import *

In [2]:
def stretch(x, y, amount=1, angle=0, center=(0,0)):
    xc, yc = center
    r = amount
    a = angle
    R = np.array([[1 + np.cos(2 * a), np.sin(2 * a), -((1 + np.cos(2 * a)) * xc + np.sin(2 * a) * yc)],
                  [np.sin(2 * a), 1 - np.cos(2 * a), -(np.sin(2 * a) * xc + (1 - np.cos(2 * a)) * yc)],
                  [0, 0, 0]])
    R = np.identity(3) + (r - 1) / 2 * R
    xn = R[0,0] * x + R[0,1] * y + R[0,2]
    yn = R[1,0] * x + R[1,1] * y + R[1,2]
    return xn, yn

def undistort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2
    q = 1 / (1 + k * r2)
    return xc + (x - xc) * q, yc + (y - yc) * q

def distort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2

    q = (1 - np.sqrt(1 - 4 * k * r2)) / (2 * k * r2 + 1e-16)
    #q = 1 / (1 + k * r2)
    return xc + (x - xc) * q, yc + (y - yc) * q

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T114503_V202411011626C_0469260100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T115108_V202411011626C_0469260125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T115708_V202411011626C_0469260150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T120308_V202411011626C_0469260175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T120908_V202411011626C_0469260200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T121508_V202411011626C_0469260225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024-09-26/solo_L1_phi-fdt-ilam_20240926T122108_V202411011626C_0469260250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2024

In [4]:
xi, yi = np.mgrid[:2048,:2048]

xd, yd = distort(xi, yi, -2e-8, (937, 1055))
xd, yd = stretch(xd, yd, 1.001, -0.567, (1023.5, 1023.5)) ### Distorted grid

dxd = xd - xi, yd - yi

In [5]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header

    image = bilinear(image, xd, yd)

    edges = find_edges(image)
    x, y = np.where(edges)
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

872.588637756036 2.3286293961977123e-05
872.453715772931 5.4253350893151975e-05
872.6723353433911 1.606705996881619e-05
872.9203232536619 5.383317943707944e-05
873.1360890979488 0.0003297497643717229
872.7335488161752 7.825742663070123e-05
873.0045663971858 3.263396707664512e-05
873.2700879741052 4.645073995013238e-05
873.6249035471662 7.077351419593203e-05


In [6]:
xi, yi = np.mgrid[:2048,:2048]

xu, yu = stretch(xi, yi, 1 / 1.001, -0.567, (1023.5, 1023.5))
xu, yu = undistort(xu, yu, -2e-8, (937, 1055)) ### Undistorted grid

dxu, dyu = xu - xi, yu - yi

In [7]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header


    edges = find_edges(image)
    x, y = np.where(edges)
    x, y = x + dxu[edges], y + dyu[edges]
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

872.6031475715431 2.2411588156745488e-05
872.4512962151488 6.188227200421004e-05
872.6779286844458 1.6626742150860352e-05
872.9262104545214 2.1949805122489785e-05
873.1362020758863 0.0003449887568972576
872.7450585390616 8.441092546640494e-05
873.0118094794776 2.6330126427609457e-05
873.273382798704 2.4542620463918752e-05
873.6054416862775 0.00013991308032901273
